In [34]:
import numpy as np
import cv2
import matplotlib as plt

## 1. HSV 색 범위
- BGR은 tracking할 때 밝기가 조금만 바뀌어도 색 값이 바뀐다.
- HSV는 **색상**(H)을 **분리해서 다루**기 때문에 **조명 변화**에 안정적이다.

In [44]:
# 일반적인 HSV에서는 360˚로 표현하지만 OpenCV에서는 0˚~180˚로 표현한다.
lower_color = np.array([100, 150, 0])
upper_color = np.array([140, 255, 255])
# lower ~ upper 의 색 범위는 파랑색 계열을 나타내고 있다.

## 2. 카메라 설정

In [45]:
# 웹캠 열기
cap = cv2.VideoCapture(0)

In [46]:
while True:
    ret, frame = cap.read()
    if not ret:
        break

    # BGR → HSV 변환
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)

    # 컬러 마스크 생성
    mask = cv2.inRange(hsv, lower_color, upper_color)

    # 윤곽선 검출
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    for cnt in contours:
        area = cv2.contourArea(cnt)
        if area > 100:  # 너무 작은 잡음은 무시
            (x, y), radius = cv2.minEnclosingCircle(cnt)
            center = (int(x), int(y))
            radius = int(radius)

            # 원으로 시각화
            cv2.circle(frame, center, radius, (0, 255, 0), 2)
            cv2.circle(frame, center, 5, (0, 0, 255), -1)  # 중심점

    # 결과 화면 출력
    cv2.imshow('Tracking Result', frame)
    cv2.imshow('Color Mask', mask)

    # 'q' 키를 누르면 종료
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# 정리
cap.release()
cv2.destroyAllWindows()

- **Object Tracking(객체 추적)**: 초록색 원이 파란 물체를 따라 움직이고 있다.  

* `cv2.findContours()`로 **색 마스크에서 윤곽선 검출**
* `cv2.minEnclosingCircle()`로 **물체 줌심과 크기 계산**
* `cv2.circle`로 **프레임에 원 시각화**
* while로 프레임 반복 → **실시간 추적처럼 보이는 효과**
